# BSV Batch Processor — Automatic Construction (text/image only)

Runs the full corpus (`data/local/pdfs` + `data/local/pdfs`),
**excluding the PDFs reserved for GT**, producing `text_image` pairs
only, no `image_caption`, no tables.

Overrides on top of the strict parser (V3), verified directly against
its real source, not assumed:
- `MIN_IMAGE_AREA = 40,000` (strict default 200,000)
- Near-white % / dominant-color % thresholds lowered from the real
  H36/H37 checks (0.80 / 0.70) to 0.45, inside `extract_images` itself,
  not a separate post-filter
- Footer filtering applied to every page, not just the last one
  (the strict parser's own top-margin filter is untouched, this adds
  a bottom-margin one alongside it)
- Table extraction disabled entirely for this run

Everything else, section detection, text extraction, pairing logic,
caption matching, stays exactly as the strict parser defines it.


In [ ]:
# Resolve paths from the repository root so notebooks work regardless of the kernel CWD.
from pathlib import Path
import os

def _find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "manifests" / "corpus_manifest.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the PhytoPairs repository root (expected data/manifests/corpus_manifest.csv)."
    )

PROJECT_ROOT = _find_project_root()
BSV_ROOT = PROJECT_ROOT / "3.Dataset Construction"
FINETUNING_ROOT = PROJECT_ROOT / "4.Finetuning"
CASE_STUDY_ROOT = PROJECT_ROOT / "5.BBCH case study"
RESULTS_RUNS_ROOT = PROJECT_ROOT / "results" / "runs" / "section_4"
DEFAULT_HF_CACHE = os.environ.get(
    "HF_HOME", str(Path.home() / ".cache" / "huggingface")
)


In [1]:
import json, os, sys, re, time, threading
import fitz, pdfplumber
from PIL import Image as PILImage
import io
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

## Compile `bsv_parser.py` from the real source notebook

Same compile step both real batches use, reading
`bsv_parser_subsections_v3.ipynb` from the working directory and
skipping its test/demo cells.


In [2]:
nb = json.load(open(str(BSV_ROOT / 'Parsers' / 'bsv_parser_subsections_v3.ipynb')))
code_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]

with open(str(BSV_ROOT / 'Parsers' / 'bsv_parser.py'), "w") as f:
    for cell in code_cells:
        src = "".join(cell["source"])
        if any(skip in src for skip in [
            "pip install",
            "parse_pdf(TEST_PDF",
            "urllib.request.urlretrieve",
            "fitz.open(TEST_PDF",
            "fitz.open(TEST_CHARENTES",
            "parse_structure(TEST_PDF",
            "os.listdir(PDF_DIR)",
            "parse_pdf(TEST_PDF2",
            "parse_pdf(TEST_CHARENTES",
            "print(SIZE_THRESHOLD)",
        ]):
            continue
        f.write(src + "\n\n")

print("Created bsv_parser.py")
print((BSV_ROOT / "Parsers" / "bsv_parser.py").is_file())

Created bsv_parser.py
True


In [3]:
if "bsv_parser" in sys.modules:
    import importlib
    import bsv_parser
    importlib.reload(bsv_parser)
sys.path.insert(0, str(BSV_ROOT / "Parsers"))
import bsv_parser
from bsv_parser import parse_pdf
print("bsv_parser loaded")

/projects/agrifutur/yabusaqe/BSV Sample Construction
['test_bulletin.pdf', '.ipynb_checkpoints', 'bsv_parser.ipynb', '__pycache__', 'PDF_URLS.txt', 'Previous Iterations Data', 'bsv_heuristics_analysis.ipynb', 'bsv_download.ipynb', 'bsv_parser_subsections.ipynb', 'bsv_parser_subsections_v2.ipynb', 'french_words.txt', 'recall_approx.py', 'reserve_gt_pdfs.py', 'gt_reserved_pdfs.jsonl', 'Old Viewers', 'bsv_batch.ipynb', 'bsv_pair_constructor.py', 'bsv_parser.py', 'bsv_parser_subsections_v3.ipynb', 'generate_reviewer.py', 'bsv_images_gt_2404', 'bsv_batch_tables_docling.ipynb', 'bsv_images_gt_tables_docling', 'bsv_dataset_gt_tables_docling.jsonl', 'bsv_pairs_gt_texttable_900.jsonl', 'generate_reviewer_V2.py', 'bsv_dataset_gt_tables_realspans.jsonl', 'processed_urls_gt_tables_docling.txt', 'bsv_pairs_gt_texttable_partial.jsonl', 'bsv_reviewer_texttable_partial.html', 'bsv_validations_export.json', 'bsv_dataset_gt_2404.jsonl', 'bsv_pairs_gt_textimage_900.jsonl', 'processed_urls_gt_2404.txt', '

## Override 1: `MIN_IMAGE_AREA`

In [4]:
bsv_parser.MIN_IMAGE_AREA = 40000   # strict default is 200,000

## Override 2: `extract_images`, near-white % / dominant-color %
lowered from 0.80/0.70 to 0.45

This is the real `extract_images` function, copied verbatim from the
strict parser, with only the two H36/H37 threshold numbers changed.
Every other check (MIN_IMAGE_BYTES, MIN_IMAGE_AREA, aspect ratio,
text-overlap, background-coverage, near-black) is untouched.


In [5]:
def loose_extract_images(pdf_path, out_img_dir, bulletin_id, images_raw, text_els=None):
    text_els = text_els or []
    all_text_els_for_page = {}
    for el in text_els:
        all_text_els_for_page.setdefault(el["page"], []).append(el)
    os.makedirs(out_img_dir, exist_ok=True)
    doc, result, seen = fitz.open(pdf_path), [], set()
    for img_info in images_raw:
        xref = img_info["xref"]
        if xref in seen:
            continue
        seen.add(xref)
        try:
            img_data = doc.extract_image(xref)
        except Exception:
            continue
        if len(img_data["image"]) < bsv_parser.MIN_IMAGE_BYTES:
            continue
        if img_data["width"] * img_data["height"] < bsv_parser.MIN_IMAGE_AREA:
            continue

        if img_data["width"] / max(img_data["height"], 1) > 5.5:
            continue  # separator strip or footer banner

        img_area = (img_info.get("right", 0) - img_info.get("left", 0)) * \
                   (img_info.get("bottom", 0) - img_info.get("top", 0))
        if img_area > 0:
            overlap_area = 0
            for el in all_text_els_for_page.get(img_info["page"], []):
                ex0, ey0 = el["left"], el["top"]
                ex1, ey1 = el["left"] + el.get("width", 0), el["top"] + el.get("size", 10)
                ix0 = max(img_info.get("left", 0), ex0)
                iy0 = max(img_info.get("top", 0), ey0)
                ix1 = min(img_info.get("right", 0), ex1)
                iy1 = min(img_info.get("bottom", 0), ey1)
                if ix1 > ix0 and iy1 > iy0:
                    overlap_area += (ix1 - ix0) * (iy1 - iy0)
            if overlap_area / img_area > 0.15:
                continue

        page_w = doc[img_info["page"] - 1].rect.width
        page_h = doc[img_info["page"] - 1].rect.height
        img_w  = img_info.get("right", 0) - img_info.get("left", 0)
        img_h  = img_info.get("bottom", 0) - img_info.get("top", 0)
        if page_w > 0 and page_h > 0:
            coverage = (img_w * img_h) / (page_w * page_h)
            if coverage > 0.40:
                continue

        pil_img = PILImage.open(io.BytesIO(img_data["image"])).convert("RGB")
        pixels = list(pil_img.getdata())
        total = len(pixels)

        # H36 (loosened): 0.80 -> 0.45
        near_white = sum(1 for r, g, b in pixels if r > 240 and g > 240 and b > 240)
        if near_white / total > 0.45:
            continue

        near_black = sum(1 for r, g, b in pixels if r < 15 and g < 15 and b < 15)
        if near_black / total > 0.80:
            continue

        # H37 (loosened): 0.70 -> 0.45
        from collections import Counter
        quantized = pil_img.quantize(colors=8)
        color_counts = Counter(quantized.getdata())
        if max(color_counts.values()) / total > 0.45:
            continue

        new_name = f"{bulletin_id}_p{img_info['page']}_{xref}.{img_data['ext']}"
        with open(os.path.join(out_img_dir, new_name), "wb") as f:
            f.write(img_data["image"])
        result.append({
            "filename": new_name,
            "page":     img_info["page"],
            "top":      img_info["top"],
            "bottom":   img_info.get("bottom", img_info["top"] + 50),
            "left":     img_info.get("left",   0),
            "right":    img_info.get("right",  1000),
            "caption":  None,
        })
    doc.close()
    return result


bsv_parser.extract_images = loose_extract_images

## Override 3: footer filtering on every page

The strict parser already drops images in the top 10% of a page
(inside `parse_pdf` itself) and skips the last page's images entirely
(inside `parse_structure`). This adds a bottom-10%-of-every-page filter
on top, without touching either of those existing checks.


In [6]:
_strict_parse_structure = bsv_parser.parse_structure   # save the real one before overwriting

def footer_filtered_parse_structure(pdf_path, table_bboxes=None):
    s_raw, i_raw, text_els = _strict_parse_structure(pdf_path, table_bboxes)

    i_raw = [
        img for img in i_raw
        if not (img.get("page_height") and img["top"] >= 0.90 * img["page_height"])
    ]

    return s_raw, i_raw, text_els

bsv_parser.parse_structure = footer_filtered_parse_structure

## Override 4: skip tables entirely for this run

In [7]:
def skip_tables(pdf_path):
    return []

bsv_parser.extract_tables = skip_tables

os.environ["SKIP_OCR"] = "1"

## Configuration

`GT_MANIFEST` is the exclusion list, this run processes every PDF in
the two real source folders **except** those reserved for GT.


In [8]:
PDF_DIRS     = [
    str(PROJECT_ROOT / "data" / "local" / "pdfs"),
    str(PROJECT_ROOT / "data" / "local" / "pdfs"),
]
GT_MANIFEST  =str(BSV_ROOT / 'Ground Truth' / 'gt_reserved_pdfs.jsonl')
OUT_JSONL    = str(BSV_ROOT / 'Automatic Construction' / 'bsv_dataset_auto_textimage.jsonl')
OUT_IMGS     = str(BSV_ROOT / 'Automatic Construction' / 'bsv_images_auto_textimage')
WORKERS      = 8
LIMIT        = None
CHECKPOINT   = str(BSV_ROOT / 'Automatic Construction' / 'Notebooks' / 'processed_urls_auto_textimage.txt')
MIN_PDF_SIZE = 5000

## Build the file list, excluding every GT-reserved PDF

In [9]:
with open(GT_MANIFEST, encoding="utf-8") as f:
    gt_reserved = set(json.loads(line)["filename"] for line in f)

print(f"{len(gt_reserved)} PDFs reserved for GT, will be excluded from this run")

pdf_paths = {}
for pdf_dir in PDF_DIRS:
    for fname in os.listdir(pdf_dir):
        if not fname.endswith(".pdf"):
            continue
        if fname in gt_reserved:
            continue
        full_path = os.path.join(pdf_dir, fname)
        if os.path.getsize(full_path) < MIN_PDF_SIZE:
            continue
        pdf_paths[fname] = full_path

pdf_files = sorted(pdf_paths.keys())
if LIMIT:
    pdf_files = pdf_files[:LIMIT]

print(f"{len(pdf_files)} PDFs to process for automatic construction")

2404 PDFs reserved for GT, will be excluded from this run
17758 PDFs to process for automatic construction


## Process, same checkpoint and threading-lock pattern as the real batches

In [10]:
def process_one(filename):
    pdf_path = pdf_paths[filename]
    if os.path.getsize(pdf_path) < MIN_PDF_SIZE:
        return filename, []
    try:
        records = parse_pdf(pdf_path, OUT_IMGS, url=None)
        return filename, records
    except Exception as e:
        return filename, []

In [11]:
os.makedirs(OUT_IMGS, exist_ok=True)
write_lock      = threading.Lock()
checkpoint_lock = threading.Lock()
counts = {"processed": 0, "sections": 0, "errors": 0}

if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        done_files = set(f.read().splitlines())
    print(f"Resuming: {len(done_files)} PDFs already processed")
else:
    done_files = set()

files_to_run = [f for f in pdf_files if f not in done_files]
print(f"Remaining: {len(files_to_run)} PDFs to process")

with open(OUT_JSONL, "a", encoding="utf-8") as out_f:
    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = {executor.submit(process_one, fname): fname for fname in files_to_run}
        for future in as_completed(futures):
            fname, records = future.result()
            with write_lock:
                for rec in records:
                    out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                counts["sections"] += len(records)
                counts["processed"] += 1
                if not records:
                    counts["errors"] += 1
            with checkpoint_lock:
                with open(CHECKPOINT, "a") as ck:
                    ck.write(fname + "\n")
            if counts["processed"] % 100 == 0 or counts["processed"] <= 5:
                print(f"[{counts['processed']}/{len(files_to_run)}] "
                      f"sections={counts['sections']} errors={counts['errors']}")

print(f"Done. Processed={counts['processed']} Sections={counts['sections']} Errors={counts['errors']}")

Resuming: 13930 PDFs already processed
Remaining: 3828 PDFs to process


/tmp/ipykernel_2194512/3406263859.py:51: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(pil_img.getdata())
/tmp/ipykernel_2194512/3406263859.py:66: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  color_counts = Counter(quantized.getdata())


[1/3828] sections=1 errors=0
[2/3828] sections=3 errors=0
[3/3828] sections=4 errors=0
[4/3828] sections=6 errors=0
[5/3828] sections=7 errors=0
[100/3828] sections=629 errors=0
[200/3828] sections=1447 errors=1
[300/3828] sections=2464 errors=1
[400/3828] sections=3356 errors=2
[500/3828] sections=3811 errors=4
[600/3828] sections=5017 errors=4
[700/3828] sections=5963 errors=4
[800/3828] sections=7168 errors=5
[900/3828] sections=8550 errors=5
[1000/3828] sections=9399 errors=5
[1100/3828] sections=9778 errors=5


Cannot set CMYK stroke color because not all values in (b'Lwn\xf6\x1b\x1d&\x02,"A<Q\x0b}vXL\xc7Z_\xae\xd2\xb9\x95\xe7s2\x9e\x8a\xa6\xf45"\xa5\xa1\xa6\x96\xfd&\xf2\x0e/+d\x1d\xb1\xd8\x10\x0c&\n\xc6\xb8\'\xcc\x1a\xd0\xcdc\x194\x13\xb3^\xd3\xbc\xc0e\x19QF\x02F(\xf1\x95\xc5\x80#\xe0^i\x86\xb9h5E(p^\x84\xa5\xf1+\xd8\xa8\xda!\x8ce\xd3Y\xc3=\xe5\xbd\x0eQ\x9a\xe49\xe3\xc2mf\xcd\xb7\xdf\x1a\x1d|\xddzs0\xff\xe1\xb0q\xa1\x10\x17V9t\xf7\xa5\xc1\x87\xf7AE\xf9\xd3\xaf\xc1\xd5\xdc\xdc\x18$\x1f\xbc\x87\x1e=6\x9f~N\xb8\xa2\x1d\xa9\xff\x18!\xadY\x18\x18F\xd9\x0b\xdb\xb0U\xc5\xe3\'\xe6n\xb6\xde\xef\xdb\xe7\x1a2K\x9bI\x1b\xae\x0e\xe1DU\xd5\xea\xa9~\xf8`\xf4\xd5\x97\xbfHf\xde\xb9\xff\xda\xee\x8bC\xde\x89H\xc2G\x17\xeb\xcf>\xf9\xf1\xe7\xef~\x03j\xf7^\xddy\xf7\xfd\xbd\xb8E\xba\xfd\xad\xc1\xf6\x8e\xe5\xfe\xe8\xe4\xe2\xfb\x83\x87\x87\xbf\x9f)\xe5NF\xe3\xd9d\xa5\x94\xae\xbd%`\x01x`+p\xe1\xfan\xe0\xeb\xc7\xfb\xda\xc3&\x01\x04\x98\xc9\xa9\xd6*\x8c\xc3lS\xb6{2N\xe1\xa6\x14\x9e *H\x8d,$;g\xbc\'Ea\xab\x95/\xd7\x1e\x

[1200/3828] sections=12042 errors=6
[1300/3828] sections=14544 errors=6
[1400/3828] sections=15323 errors=6
[1500/3828] sections=16128 errors=6
[1600/3828] sections=17504 errors=6
[1700/3828] sections=18929 errors=7
[1800/3828] sections=19957 errors=7
[1900/3828] sections=21006 errors=7
[2000/3828] sections=22094 errors=7
[2100/3828] sections=23343 errors=7
[2200/3828] sections=24392 errors=7
[2300/3828] sections=24751 errors=7
[2400/3828] sections=25410 errors=7
[2500/3828] sections=26138 errors=7
[2600/3828] sections=26477 errors=7
[2700/3828] sections=26781 errors=7
[2800/3828] sections=27065 errors=7
[2900/3828] sections=27980 errors=7
[3000/3828] sections=29100 errors=7
[3100/3828] sections=30990 errors=7
[3200/3828] sections=32991 errors=7


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


[3300/3828] sections=34809 errors=7
[3400/3828] sections=35714 errors=7


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Execute called on non-indirect object (inline image?) <PDFStream(None): raw=0, {}>
Execute called on non-indirect object (inline image?) <PDFStream(None): raw=0, {}>
Execute called on non-indirect object (inline image?) <PDFStream(None): raw=0, {}>
Execute called on non-indirect object (inline image?) <PDFStream(None): raw=0, {}>
Execute called on non-indirect object (inline image?) <PDFStream(N

[3500/3828] sections=36778 errors=7


Data-loss while decompressing corrupted data


[3600/3828] sections=37951 errors=7
[3700/3828] sections=39599 errors=7
[3800/3828] sections=40482 errors=8
Done. Processed=3828 Sections=40623 Errors=9


## Build text_image pairs only

In [12]:
sys.path.insert(0, str(BSV_ROOT / "Scripts"))
from bsv_pair_constructor import generate_pairs

OUT_PAIRS = str(BSV_ROOT / 'Automatic Construction' / 'bsv_pairs_auto_textimage.jsonl')

all_pairs = []
with open(OUT_JSONL, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        pairs = generate_pairs(rec)
        all_pairs.extend(p for p in pairs if p["pair_type"] == "text_image")

print(f"{len(all_pairs)} text_image pairs built")

with open(OUT_PAIRS, "w", encoding="utf-8") as f:
    for p in all_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"Written to {OUT_PAIRS}")

34448 text_image pairs built
Written to /projects/agrifutur/yabusaqe/BSV Sample Construction/bsv_pairs_auto_textimage.jsonl
